In [183]:
import kagglehub
import matplotlib.pyplot as plt
import os

In [184]:
path = kagglehub.dataset_download('techsash/waste-classification-data')
os.listdir(f"{path}/DATASET")

['TEST', 'TRAIN']

In [185]:
dataset_path = f"{path}/DATASET/TRAIN"
garbage_types = os.listdir(dataset_path)
print(garbage_types)

['O', 'R']


In [186]:
from PIL import Image
train_df = []
labels = []
dimensions = set()

for type in garbage_types:
    folder_path = os.path.join(dataset_path,type) 

    if os.path.isdir(folder_path):
        images = [f for f in os.listdir(folder_path)  if f.endswith(('jpeg','jpg','png'))]
        for i in images:
            train_df.append(folder_path + '/' + i)
        image_len = len(images)

        print(f"{type} type contains {image_len} image.")

        for image_file in images:
            image_path = os.path.join(folder_path, image_file)
            with Image.open(image_path) as img:
                widht,height = img.size
                channels = len(img.getbands())
                dimensions.add((widht,height,channels))
        print(f"{type} type image dimensions are {dimensions.pop()}")
len(dimensions)

O type contains 12565 image.
O type image dimensions are (263, 192, 3)
R type contains 9999 image.
R type image dimensions are (190, 190, 3)


960

In [187]:
labels.clear()
for i in range(12565):
    labels.append('O')
for j in range(9999):
    labels.append('R')
print(labels)
len(labels)

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O',

22564

In [188]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# Örnek bir DataFrame oluşturma (gerçek veriyle değiştirebilirsiniz)
data = {
    'filepath': train_df,   # Resim yolları
    'label': labels  # Etiketler (örneğin: 0 ve 1 gibi)
}
train_df = pd.DataFrame(data)


#train_df = train_df.sample(n=10000, random_state=42)  # 10000 rastgele örnek seçer
print(len(train_df))

# ImageDataGenerator'ı başlat
train_data_gen = ImageDataGenerator(rescale=1./255)  # Resimleri [0, 1] aralığına normalize et

# flow_from_dataframe ile veri yükleme
train_generator = train_data_gen.flow_from_dataframe(
    dataframe=train_df,         # DataFrame
    x_col='filepath',           # Resim dosyalarının bulunduğu sütun
    y_col='label',              # Etiketlerin bulunduğu sütun
    target_size=(384, 384),     # Resimleri 384x384 boyutuna yeniden boyutlandır
    batch_size=128,              # Batch boyutu
    class_mode='categorical',   # Kategorik etiketler
    seed=42,                    # Rastgelelikik tohumunu belirle
    shuffle=True                # Verileri karıştır
)


22564
Found 22564 validated image filenames belonging to 2 classes.


In [189]:
train_data_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [190]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Basit bir CNN modeli oluşturma
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(384, 384, 3)),  # Giriş katmanı
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')  # Kategorik etiketler için softmax kullanıyoruz
])

c:\Users\FURKAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [191]:
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_24 (Conv2D)              │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_25 (MaxPooling2D) │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 565504)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │    36,192,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,211,842 (138.14 MB)

 Trainable params: 36,211,842 (138.14 MB)

 Non-trainable params: 0 (0.00 B)

In [192]:
# Modeli derleme
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [193]:
# Modeli eğitme
model.fit(
    train_generator,          # Veri üreticisi
    epochs=10,                # Eğitim dönemi sayısı
    steps_per_epoch=len(train_generator),  # Bir epoch'taki adım sayısı
)

c:\Users\FURKAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 607s 3s/step - accuracy: 0.6827 - loss: 5.0810
Epoch 2/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 630s 4s/step - accuracy: 0.8338 - loss: 0.3886
Epoch 3/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 738s 4s/step - accuracy: 0.8708 - loss: 0.3046
Epoch 4/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 631s 4s/step - accuracy: 0.9119 - loss: 0.2160
Epoch 5/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 633s 4s/step - accuracy: 0.9452 - loss: 0.1418
Epoch 6/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 645s 4s/step - accuracy: 0.9684 - loss: 0.0881
Epoch 7/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 599s 3s/step - accuracy: 0.9803 - loss: 0.0589
Epoch 8/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 679s 4s/step - accuracy: 0.9861 - loss: 0.0425
Epoch 9/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 575s 3s/step - accuracy: 0.9903 - loss: 0.0319
Epoch 10/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 625s 4s/step - accuracy: 0.9870 - loss: 0.0393


In [194]:
testdata_path = f"{path}/DATASET/TEST"
test_df = []
test_labels = []

for type in garbage_types:
    folder_path = os.path.join(testdata_path,type) 

    if os.path.isdir(folder_path):
        images = [f for f in os.listdir(folder_path)  if f.endswith(('jpeg','jpg','png'))]
        for i in images:
            test_df.append(folder_path + '/' + i)
        image_len = len(images)

        print(f"{type} type contains {image_len} image.")

        for image_file in images:
            image_path = os.path.join(folder_path, image_file)
            with Image.open(image_path) as img:
                widht,height = img.size
                channels = len(img.getbands())
                dimensions.add((widht,height,channels))
        print(f"{type} type image dimensions are {dimensions.pop()}")
len(dimensions)
len(test_df)

O type contains 1401 image.
O type image dimensions are (432, 116, 3)
R type contains 1112 image.
R type image dimensions are (249, 203, 1)


2513

In [195]:
test_labels.clear()
for i in range(1401):
    test_labels.append('O')
for j in range(1112):
    test_labels.append('R')
print(test_labels)
len(test_labels)

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O',

2513

In [196]:
# DataFrame oluşturma
testimages_df = pd.DataFrame({
    'filepath': test_df,  # Resim dosyalarının yolu
    'label': test_labels         # Etiketler
})
# Test verisi için ImageDataGenerator oluşturma (veri artırma yapmadan)
test_data_gen = ImageDataGenerator(rescale=1./255)

# Test verisini yükleme
test_generator = test_data_gen.flow_from_dataframe(
    dataframe=testimages_df,          # Test verisinin bulunduğu DataFrame
    x_col='filepath',           # Resim dosyalarının bulunduğu sütun
    y_col='label',              # Etiketlerin bulunduğu sütun
    target_size=(384, 384),     # Resimleri 384x384 boyutuna yeniden boyutlandır
    batch_size=32,              # Batch boyutu
    class_mode='categorical',   # Kategorik etiketler
    shuffle=False,              # Test verisini karıştırma
    seed=42                     # Rastgelelikik tohumunu belirle
)

Found 2513 validated image filenames belonging to 2 classes.


In [197]:
loss,accuracy = model.evaluate(test_generator)
print(f"Loss:{loss}\nAccuracy:{accuracy}")

c:\Users\FURKAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 185ms/step - accuracy: 0.9025 - loss: 0.5099
Loss:0.6975261569023132
Accuracy:0.8416235446929932
